# Node 05 · V3 MD 系统制备 —— 批处理（33 体系） · v0.2.0

单分子跑通已在 `pool_mol_030`（+2、41 重原子）上证明机制。此为批处理版：
I1 本地建配体 → 生成每体系自包含 `run_prep.sh` + 两阶段 Slurm 数组 → 提交 → 回收并逐体系 fail-closed 断言。

## 六道不变量

| | 强制点 | 断言 |
|---|---|---|
| **I1** 配体几何来自 master SDF + 姿势坐标 | 替换 V2 step6b | 重建 SMILES 图不变 · 重原子键长守恒 · 形式电荷 == frozen |
| **I2** ACPYPE 只用 frozen_net_charge（禁扫描） | 替换 V2 step6c 参数化 | `ligand_GMX.itp` 部分电荷和 == frozen_net_charge |
| **I2b** 配体分子图不变 【v0.2.0 新增】 | SDF → 参数化之间 | itp 的 `[atoms]`/`[bonds]` 重建的元素序列与邻接 == 投喂的 `ligand.sdf` |
| **I3** 中性二联体（HIE His41 + 中性 CYS） | pdb2gmx | `protein.gro` 中 His41=HISE、Cys145=CYS |
| **I3b** 残基序列全局一致 【v0.2.0 新增】 | pdb2gmx | 输入 PDB 与 `protein.gro` 残基序列逐位相等；全部 His 为 HISE |
| **I4** 反离子由 (protein + frozen) 推导 | 溶剂化 / genion | grompp 前感知电荷 == 推导值（无证据即判失败）· genion 后系统恰中性 |

**为何需要 I2b**：I2 只断言部分电荷之和，而 AM1-BCC 电荷【按构造】就求和为 `-nc`——
该和对键级、芳香性判定、逐原子形式电荷的任何改动完全不敏感。V2 所损坏的那类分子会带着正确的
`itp_charge_sum` 通过 I2。I2b 闭合 master → MD 拓扑的最后一跳。

**为何需要 I3b**：pdb2gmx 可能重新编号，而 I3 只读某一索引处的残基名；Mpro 有多个 His / Cys，
编号偏移时存在巧合通过的可能。I3b 一次性验证全局编号与组氨酸统一性。

## 已锁定的蛋白侧值（跑通捕获，对 6W63 全部分子通用）
`PROTEIN_CHARGE_6W63 = -4` · His 7 个全 HIE · 受体 `data/receptors/6W63_protein_only.pdb`

## 执行顺序（不可省）
1. Cell 1–3（本地，仅 RDKit）
2. Cell 4 生成脚本 → **先交互式冒烟测试一份 `run_prep.sh`**（约 15 min）
3. 通过后提交数组 A（31 primary），A 成功后由 `--dependency=afterok` 触发数组 B（2 secondary）
4. Cell 6 回收 + 六道断言 + 受体跨体系同一性

**次姿势必须晚于其主姿势**：secondary 体系复用 primary 的 `ligand_GMX.itp`
（使姿势比较不混入电荷参数化差异，AM1-BCC 电荷依赖构象），故不可与之并发。


In [1]:
# Cell 1 · 配置（v0.2.0）
from pathlib import Path
import sys, json, importlib
PR = Path("/mnt/hpc/home/wanghongmei/lang2").resolve()
if str(PR) not in sys.path: sys.path.insert(0, str(PR))
PY    = "/mnt/hpc/home/wanghongmei/anaconda3/envs/md_env/bin/python"
GMXRC = "/mnt/hpc/home/wanghongmei/apps/gromacs-2025.4-cuda12.1/bin/GMXRC"

TARGET              = "6W63"
RECEPTOR_PDB        = PR/"data"/"receptors"/f"{TARGET}_protein_only.pdb"   # Node 04a 清洗输出
PROTEIN_CHARGE_6W63 = -4                       # ★ 跑通捕获（pdb2gmx 报告；统一中性 His+Cys）

N02_SDF     = PR/"results"/"02_master_molecules"/"sdf"
N03_MAPS    = PR/"results"/"03_ligand_pdbqt"/"atom_maps"
N04_POSES   = PR/"results"/"04_docking"/"exhaustiveness_arm"/"poses"/TARGET
N04_SEL     = PR/"results"/"04_docking"/"Node04_md_selection.json"
N04_POSESEL = PR/"results"/"04_docking"/f"Node04_pose_selection_{TARGET}.json"
OUT_ROOT    = PR/"md_solv_v3"; (OUT_ROOT/"logs").mkdir(parents=True, exist_ok=True)

from core import node05_md_prep as N5; importlib.reload(N5)
assert N5.NODE05_VERSION >= "0.2.0", f"需 v0.2.0（含 I2b/I3b），当前 {N5.NODE05_VERSION}"

# v0.2.0：His 应答由受体实测推导并断言计数（跨靶点臂 His 数不同，不可硬编码 7）
HIS_ANSWERS, HIS_RESIDS = N5.his_answers_for_receptor(str(RECEPTOR_PDB), expected_n=7)

print("node05_md_prep", N5.NODE05_VERSION)
for p in [RECEPTOR_PDB, N02_SDF, N03_MAPS, N04_POSES, N04_SEL, N04_POSESEL]:
    print(f"  {'OK ' if Path(p).exists() else 'MISSING'}  {p}")
print("蛋白电荷:", PROTEIN_CHARGE_6W63, "| His 残基:", HIS_RESIDS, "| 应答:", repr(HIS_ANSWERS))

node05_md_prep 0.2.0
  OK   /mnt/hpc/home/wanghongmei/lang2/data/receptors/6W63_protein_only.pdb
  OK   /mnt/hpc/home/wanghongmei/lang2/results/02_master_molecules/sdf
  OK   /mnt/hpc/home/wanghongmei/lang2/results/03_ligand_pdbqt/atom_maps
  OK   /mnt/hpc/home/wanghongmei/lang2/results/04_docking/exhaustiveness_arm/poses/6W63
  OK   /mnt/hpc/home/wanghongmei/lang2/results/04_docking/Node04_md_selection.json
  OK   /mnt/hpc/home/wanghongmei/lang2/results/04_docking/Node04_pose_selection_6W63.json
蛋白电荷: -4 | His 残基: [41, 64, 80, 163, 164, 172, 246] | 应答: '1\n1\n1\n1\n1\n1\n1\n'


In [2]:
# Cell 2 · 载入 Node 04 选择 → 33 体系（medoid 种子取自 pose_selection）
sel  = json.loads(Path(N04_SEL).read_text())
psel = json.loads(Path(N04_POSESEL).read_text())
rows = sel["selected"]                               # {ligand, formal_charge, ...}
pose_by_lig = {r["ligand"]: r for r in psel["selection"]}

# —— 姿势来源（Methods 须明写：选择在 exh=32 下做出，轨迹起自 exh=128 姿势）——
print("pose_source     :", psel["pose_source"])
print("primary rule    :", psel["rule"]["primary"])
print("secondary rule  :", psel["rule"]["secondary"])
print("pre_registered  :", psel["rule"]["pre_registered"],
      "| n_complexes", psel["n_complexes"], "| n_with_secondary", psel["n_with_secondary"])

def poses_for(name):
    r = pose_by_lig[name]
    out = [{"pose_id": "primary", "seed": int(r["primary_seed"]), "model": 1}]
    if r.get("secondary_seed") is not None:          # 仅 pool_mol_011 / _020
        out.append({"pose_id": "secondary", "seed": int(r["secondary_seed"]), "model": 1})
    return out

jobs = []
for rec in rows:
    name, q = rec["ligand"], int(rec["formal_charge"])
    for pz in poses_for(name):
        jobs.append({"name": name, "target": TARGET, "frozen_net_charge": q,
                     "pose_id": pz["pose_id"], "seed": pz["seed"], "model": pz["model"]})

n_sec = sum(1 for j in jobs if j["pose_id"] == "secondary")
assert len(jobs) == 33 and n_sec == 2, f"应为 33 体系（31 primary + 2 secondary），实得 {len(jobs)}/{n_sec}"
print(f"\n体系 {len(jobs)}（31 primary + {n_sec} secondary）→ 轨迹 {len(jobs)*3}")
print("示例:", jobs[0])
print("次姿势:", [j["name"] for j in jobs if j["pose_id"] == "secondary"])

pose_source     : exhaustiveness_arm (exhaustiveness=128)
primary rule    : best-scoring member of the largest cluster
secondary rule  : best-scoring member of cluster 2, only if it holds >=2 seeds
pre_registered  : True | n_complexes 31 | n_with_secondary 2

体系 33（31 primary + 2 secondary）→ 轨迹 99
示例: {'name': 'pool_mol_001', 'target': '6W63', 'frozen_net_charge': 0, 'pose_id': 'primary', 'seed': 46, 'model': 1}
次姿势: ['pool_mol_011', 'pool_mol_020']


In [3]:
# Cell 3 · I1（本地 RDKit）：master SDF + 姿势坐标 → 每体系 ligand.sdf
# v0.2.0：bond_tol 0.15 → 0.02（pool_mol_030 实测 0.001 Å；Vina 不改变键长，
#         0.15 丢弃了本检查约 150 倍分辨力）；请求的 MODEL 不存在时 fail-closed。
import statistics

i1_all, stale = {}, []
for job in jobs:
    name, pid = job["name"], job["pose_id"]
    SYS = OUT_ROOT/f"{name}__{TARGET}__{pid}"; SYS.mkdir(parents=True, exist_ok=True)
    if (SYS/"ligand.acpype").exists():
        stale.append(SYS.name)
    try:
        i1_all[(name, pid)] = N5.apply_pose_to_master(
            str(N02_SDF/f"{name}.sdf"),
            str(N04_POSES/f"{name}__{TARGET}__s{job['seed']}.pdbqt"),
            str(N03_MAPS/f"{name}.json"), job["frozen_net_charge"],
            str(SYS/"ligand.mol2"), model=job["model"])
    except Exception as e:
        i1_all[(name, pid)] = {"ok": False, "error": f"{type(e).__name__}: {e}"}

ok = sum(1 for v in i1_all.values() if v.get("ok"))
print(f"I1 通过 {ok}/{len(jobs)}")
for k, v in i1_all.items():
    if not v.get("ok"):
        print("  FAIL", k, v.get("error", "")[:110])

# 键长偏差分布 —— 这才是 I1 的证据；仅看通过计数几乎不携带信息
devs = sorted(v["max_heavy_bond_dev_A"] for v in i1_all.values() if v.get("ok"))
if devs:
    print(f"max_heavy_bond_dev_A: min {devs[0]:.4f} / median {statistics.median(devs):.4f} "
          f"/ max {devs[-1]:.4f}   (容差 0.02)")

if stale:
    print(f"\n注意：{len(stale)} 个体系目录已存在 ligand.acpype/（跑通或先前运行遗留）。")
    print("  run_prep.sh 开头会 rm -rf ligand.acpype 后重建，故批处理路径安全；")
    print("  但在此之前【不要】运行 Cell 6——那会读到旧姿势参数化的 itp。")
    print("  样例:", stale[:3])

assert ok == len(jobs), "I1 有失败 —— 修复后再生成脚本"

I1 通过 33/33
max_heavy_bond_dev_A: min 0.0010 / median 0.0010 / max 0.0020   (容差 0.02)

注意：2 个体系目录已存在 ligand.acpype/（跑通或先前运行遗留）。
  run_prep.sh 开头会 rm -rf ligand.acpype 后重建，故批处理路径安全；
  但在此之前【不要】运行 Cell 6——那会读到旧姿势参数化的 itp。
  样例: ['pool_mol_001__6W63__primary', 'pool_mol_030__6W63__primary']


In [4]:
# Cell 4 · 生成每体系 run_prep.sh + 两阶段 Slurm 数组（不在此执行）
importlib.reload(N5)
print("node05_md_prep", N5.NODE05_VERSION)

def sysdir(j):
    return OUT_ROOT/f"{j['name']}__{TARGET}__{j['pose_id']}"

primary   = [j for j in jobs if j["pose_id"] == "primary"]
secondary = [j for j in jobs if j["pose_id"] != "primary"]
assert len(primary) == 31 and len(secondary) == 2, (len(primary), len(secondary))

def write_scripts(job_list):
    paths = []
    for job in job_list:
        SYS   = sysdir(job)
        geom  = Path(i1_all[(job["name"], job["pose_id"])]["ligand_geom_file"]).name
        reuse = (str(OUT_ROOT/f"{job['name']}__{TARGET}__primary")
                 if job["pose_id"] != "primary" else None)
        (SYS/"run_prep.sh").write_text(
            N5.emit_run_prep_script(str(SYS), str(RECEPTOR_PDB), TARGET,
                                    job["frozen_net_charge"], geom, HIS_ANSWERS,
                                    str(PR), GMXRC, PY, reuse_itp_from=reuse))
        paths.append(str(SYS/"run_prep.sh"))
    return paths

p_paths, s_paths = write_scripts(primary), write_scripts(secondary)
(OUT_ROOT/"prep_scripts_primary.txt").write_text("\n".join(p_paths) + "\n")
(OUT_ROOT/"prep_scripts_secondary.txt").write_text("\n".join(s_paths) + "\n")
(OUT_ROOT/"node05_prep_A.sbatch").write_text(
    N5.emit_array_sbatch(str(OUT_ROOT/"prep_scripts_primary.txt"),
                         len(p_paths), str(PR), str(OUT_ROOT/"logs")))
(OUT_ROOT/"node05_prep_B.sbatch").write_text(
    N5.emit_array_sbatch(str(OUT_ROOT/"prep_scripts_secondary.txt"),
                         len(s_paths), str(PR), str(OUT_ROOT/"logs")))

print(f"已生成 {len(p_paths)} + {len(s_paths)} 份 run_prep.sh；两个数组已写出")
print(f"""
① 先冒烟测试一份（约 15 min；这是在不烧掉 33 个任务的前提下确认
   `source GMXRC` 能否熬过 `set -u`、以及 antechamber/acpype 路径的唯一办法）：
     cd {PR}
     bash {p_paths[0]}

② 通过后按顺序提交（B 依赖 A 成功；secondary 复用 primary 的 itp，不可并发）：
     cd {PR}
     A=$(sbatch --parsable {OUT_ROOT}/node05_prep_A.sbatch)
     echo "array A = $A"
     sbatch --dependency=afterok:$A {OUT_ROOT}/node05_prep_B.sbatch

③ 全部完成后回到 Cell 6 回收 + 六道断言。
""")

node05_md_prep 0.2.0
已生成 31 + 2 份 run_prep.sh；两个数组已写出

① 先冒烟测试一份（约 15 min；这是在不烧掉 33 个任务的前提下确认
   `source GMXRC` 能否熬过 `set -u`、以及 antechamber/acpype 路径的唯一办法）：
     cd /mnt/hpc/home/wanghongmei/lang2
     bash /mnt/hpc/home/wanghongmei/lang2/md_solv_v3/pool_mol_001__6W63__primary/run_prep.sh

② 通过后按顺序提交（B 依赖 A 成功；secondary 复用 primary 的 itp，不可并发）：
     cd /mnt/hpc/home/wanghongmei/lang2
     A=$(sbatch --parsable /mnt/hpc/home/wanghongmei/lang2/md_solv_v3/node05_prep_A.sbatch)
     echo "array A = $A"
     sbatch --dependency=afterok:$A /mnt/hpc/home/wanghongmei/lang2/md_solv_v3/node05_prep_B.sbatch

③ 全部完成后回到 Cell 6 回收 + 六道断言。



In [5]:
# 提交前单体系验证（pool_mol_001，q=0 → derived=-4，非空洞比较）
S = OUT_ROOT/"pool_mol_001__6W63__primary"
q = 0
pre = (S/"grompp_pre.log").read_text()
print("charge line:", [l for l in pre.splitlines() if "total charge" in l.lower()])

print("I2 :", N5.assert_ligand_charge(str(S/"ligand.acpype"/"ligand_GMX.itp"), q))
print("I2b:", N5.assert_ligand_graph(str(S/"ligand.acpype"/"ligand_GMX.itp"), str(S/"ligand.sdf")))
print("I3 :", N5.assert_dyad_states(str(S/"protein.gro"), TARGET))
print("I3b:", N5.assert_residue_sequence(str(RECEPTOR_PDB), str(S/"protein.gro")))
print("I4 :", N5.assert_counterions(PROTEIN_CHARGE_6W63, q, pre, str(S/"complex.gro")))

charge line: ['  System has non-zero total charge: -4.000000', '  Total charge should normally be an integer. See']
I2 : {'invariant': 'I2_frozen_charge', 'ok': True, 'itp_charge_sum': 0.0, 'frozen_net_charge': 0, 'n_atoms': 55}
I2b: {'invariant': 'I2b_ligand_graph', 'ok': True, 'n_atoms': 55, 'n_bonds': 59, 'element_counts': {'C': 28, 'N': 4, 'O': 3, 'H': 20}, 'atom_order_preserved': True, 'ligand_sdf_sha256': '1e55d3071e46b2b0a4718978fcb06cccd8f8ee815a1a1c32f9a97acba8090a40'}
I3 : {'invariant': 'I3_neutral_dyad', 'ok': True, 'target': '6W63', 'His41': 'HISE', 'Cys145': 'CYS', 'note': 'HISE==HIE（ε-质子化中性），GROMACS 命名'}
I3b: {'invariant': 'I3b_residue_sequence', 'ok': True, 'n_residues': 305, 'renumbered_by_pdb2gmx': False, 'n_his': 7, 'his_states': ['HISE'], 'n_cys': 12}
I4 : {'invariant': 'I4_derived_counterions', 'ok': True, 'protein_charge': -4, 'frozen_net_charge': 0, 'derived_system_charge': -4, 'grompp_perceived': -4, 'n_NA': 90, 'n_CL': 86, 'final_net_charge': 0}


In [6]:
import numpy as np
from rdkit import Chem
S   = OUT_ROOT/"pool_mol_001__6W63__primary"
m   = Chem.MolFromMolFile(str(S/"ligand.sdf"), removeHs=False)
sdf = m.GetConformer().GetPositions()
L   = (S/"ligand.acpype"/"ligand_GMX.gro").read_text().splitlines(); n = int(L[1])
gro = np.array([[float(l[20:28]), float(l[28:36]), float(l[36:44])] for l in L[2:2+n]]) * 10.0
_, bonds = N5.parse_itp_atoms_bonds(str(S/"ligand.acpype"/"ligand_GMX.itp"))
dev = max(abs(np.linalg.norm(sdf[i]-sdf[j]) - np.linalg.norm(gro[i]-gro[j]))
          for i, j in (tuple(b) for b in bonds))
print(f"n_atoms gro={n} sdf={len(sdf)} | 键长最大偏差 gro vs sdf: {dev:.4f} Å")

n_atoms gro=55 sdf=55 | 键长最大偏差 gro vs sdf: 0.0087 Å


In [9]:
sdf_q = np.round(sdf/10, 3)*10                      # 把 sdf 量化到 .gro 精度
dev_q = max(abs(np.linalg.norm(sdf_q[i]-sdf_q[j]) - np.linalg.norm(gro[i]-gro[j]))
            for i, j in (tuple(b) for b in bonds))
print(f"量化后偏差: {dev_q:.6f} Å   (应 ~1e-6)")

量化后偏差: 0.009912 Å   (应 ~1e-6)


In [11]:
# Cell 5 ·（可选）监控数组进度
r = N5.run("squeue -u $USER -n node05_prep -h | wc -l")
print("排队/运行中的 prep 任务:", r.stdout.strip())
done    = [j for j in jobs if (sysdir(j)/"complex.gro").exists()]
missing = [sysdir(j).name for j in jobs if not (sysdir(j)/"complex.gro").exists()]
print(f"已产出 complex.gro 的体系: {len(done)}/{len(jobs)}")
if missing:
    print("未完成:", missing[:8], "..." if len(missing) > 8 else "")
    print("查失败原因:  tail -40", OUT_ROOT/"logs"/"prep_<jobid>_<task>.err")

排队/运行中的 prep 任务: 0
已产出 complex.gro 的体系: 33/33


In [15]:
# Cell 6 · 回收 + 逐体系六道断言 fail-closed + 受体跨体系同一性 + manifest
# 重跑 Cell 6 的最后一段即可，无需重跑上面的 33 体系断言
importlib.reload(N5); print("N5", N5.NODE05_VERSION)          # 应为 0.2.1
rid = N5.assert_receptor_topology_identical([str(sysdir(j)) for j in jobs])
print(json.dumps(rid, indent=1, ensure_ascii=False))



import pandas as pd
importlib.reload(N5)

summary = []
for job in jobs:
    name, pid, q = job["name"], job["pose_id"], job["frozen_net_charge"]
    SYS = OUT_ROOT/f"{name}__{TARGET}__{pid}"
    rec = {"molecule": name, "pose": pid, "q": q}
    try:
        itp = SYS/"ligand.acpype"/"ligand_GMX.itp"
        lig = SYS/Path(i1_all[(name, pid)]["ligand_geom_file"]).name
        pre = (SYS/"grompp_pre.log").read_text() if (SYS/"grompp_pre.log").exists() else ""
        inv = {
            "I1":  i1_all[(name, pid)],
            "I2":  N5.assert_ligand_charge(str(itp), q),
            "I2b": N5.assert_ligand_graph(str(itp), str(lig)),
            "I3":  N5.assert_dyad_states(str(SYS/"protein.gro"), TARGET),
            "I3b": N5.assert_residue_sequence(str(RECEPTOR_PDB), str(SYS/"protein.gro")),
            "I4":  N5.assert_counterions(PROTEIN_CHARGE_6W63, q, pre, str(SYS/"complex.gro")),
        }
        man = N5.prep_manifest_v2(job, inv, SYS)
        rec.update({"passed": man["all_passed"],
                    "dev_A":  round(float(inv["I1"]["max_heavy_bond_dev_A"]), 4),
                    "atoms":  inv["I2b"]["n_atoms"], "bonds": inv["I2b"]["n_bonds"],
                    "NA": inv["I4"]["n_NA"], "CL": inv["I4"]["n_CL"],
                    "sys_q": inv["I4"]["derived_system_charge"],
                    "net": inv["I4"]["final_net_charge"]})
    except Exception as e:
        rec.update({"passed": False, "error": f"{type(e).__name__}: {e}"[:100]})
    summary.append(rec)

df = pd.DataFrame(summary)
print(df.to_string(index=False))
npass = int(df["passed"].sum())
print(f"\n六道断言全过体系: {npass}/{len(df)} → 轨迹 {npass*3}")

if npass:
    d = df.loc[df.passed, "dev_A"].astype(float)
    print(f"max_heavy_bond_dev_A: min {d.min():.4f} / median {d.median():.4f} / max {d.max():.4f}"
          f"   (容差 0.02)")

if npass == len(df):
    rid = N5.assert_receptor_topology_identical([str(sysdir(j)) for j in jobs])
    print("受体拓扑跨 33 体系同一 · protein.gro sha256(excl title) =",
          rid["files"]["protein.gro"]["sha256_excl_title"][:16], "...")    
    print("\n全部体系制备通过。下一步：MDP 层（v0.2.1，待 V2 .mdp 上传后再定）。")
    print("请把本 Cell 的表格与上面两行发回复核，再启动 replica。")
else:
    print("\n有体系未过 —— 逐条看 error（真实参数化问题，勿绕过；勿改电荷绕行 = I2 的意义）。")
    print(df.loc[~df.passed, ["molecule", "pose", "q", "error"]].to_string(index=False))

N5 0.2.1
{
 "invariant": "receptor_topology_identical",
 "ok": true,
 "files": {
  "posre_prot.itp": {
   "sha256": "ddf037df6b91c10de45aa68261bfbe3df41ceee00ffd190c73e68abc9120103e",
   "n_systems": 33
  },
  "protein.gro": {
   "sha256_excl_title": "61dc910cf99025980104342d8647c6b32ed9884f5788cefd9c8a7e6cba7f65b2",
   "n_systems": 33,
   "max_coord_dev_angstrom": 0.0,
   "n_distinct_gromacs_quotes": 19
  }
 }
}
    molecule      pose  q  passed  dev_A  atoms  bonds  NA  CL  sys_q  net
pool_mol_001   primary  0    True  0.001     55     59  90  86     -4    0
pool_mol_003   primary  0    True  0.001     51     54  90  86     -4    0
pool_mol_004   primary  0    True  0.002     51     55  90  86     -4    0
pool_mol_005   primary  0    True  0.002     49     53  90  86     -4    0
pool_mol_006   primary  0    True  0.001     61     64  90  86     -4    0
pool_mol_007   primary  0    True  0.001     62     65  90  86     -4    0
pool_mol_009   primary  0    True  0.001     61     64  90

In [16]:
# Node 05a 冻结：汇总 33 份 manifest + 受体同一性 + 上游锚点
import hashlib, datetime, json
freeze = {
    "module": "node05_md_prep", "version": N5.NODE05_VERSION,
    "created_at_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "target": TARGET, "n_systems": len(jobs), "n_trajectories": len(jobs) * 3,
    "receptor": {"clean_pdb": str(RECEPTOR_PDB),
                 "clean_pdb_sha256": N5.sha256_file(RECEPTOR_PDB),
                 "protein_charge": PROTEIN_CHARGE_6W63,
                 "his_residues": HIS_RESIDS, "his_state": "all HISE (epsilon)",
                 **rid["files"]},
    "upstream": {"md_selection": N5.sha256_file(N04_SEL),
                 "pose_selection": N5.sha256_file(N04_POSESEL),
                 "pose_source": psel["pose_source"]},
    "systems": [json.loads((sysdir(j) / "node05_prep_manifest.json").read_text())
                for j in jobs],
}
p = OUT_ROOT / "Node05_prep_freeze.json"
p.write_text(json.dumps(freeze, indent=1, ensure_ascii=False))
print(p, "|", N5.sha256_file(p))

/mnt/hpc/home/wanghongmei/lang2/md_solv_v3/Node05_prep_freeze.json | c81f4fa0ddac55fef4623f31a422409f93c8a41153b83c0e6b13f9d5447261f4


---

## v0.2.0 未触及、仍待办（MDP 层）

以下各项须先上传 V2 的各 `.mdp`、`core/step6c_run_gromacs.py`、`run_md.sh` / `run_md2.sh`
之后才处理，不在本 notebook 范围内：

| 项 | 现状 | 建议 |
|---|---|---|
| `write_mdp()` | 已定义，但两个 notebook 均未调用；参数系凭记忆重建 | 读 V2 `.mdp` 后据实重写 |
| replica 种子 | `gen-seed = -1`（由时钟/PID 取），不可复现、未记录 | 改显式互异种子并写入 manifest（如 Node 04 的 42–46） |
| 配体位置限制 | `-DPOSRES` 只约束蛋白；配体在 NVT/NPT 期间自由 | 引入 ACPYPE 已产出的 `posre_ligand.itp`（`POSRESLIG`） |
| `refcoord-scaling` | `-DPOSRES` + Parrinello-Rahman 下缺失，警告被 `-maxwarn 5` 掩盖 | 加 `refcoord-scaling = com`；平衡阶段考虑 C-rescale |
| `tc-grps` | `System` 单组耦合 | 建议两组（`Protein_LIG` / `Water_and_ions`）—— 待张老师确认 |

## Methods 须明写的两点

1. **姿势来源与选择口径不同层**：31 分子的选择在 `exhaustiveness = 32` 的主对接臂下做出，
   而 MD 轨迹起自 `exhaustiveness = 128` 对照臂的姿势（`pose_source` 字段已记录）。
2. **AM1-BCC 电荷拟合于对接构象**；次姿势体系复用主姿势的电荷集，
   使姿势比较不混入电荷参数化差异。
